In [2]:
# Experiment 7: Next-token probabilities - inspect logits and top-k candidates from a small open model (GPT-2)
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def top_k_next(prompt, k=1):
    ids = tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids).logits[0, -1]          # scores for the NEXT token only
    probs = torch.softmax(logits, dim=-1)          # logits -> probability distribution
    p, idx = probs.topk(k)
    return [{"prompt": prompt, "rank": r + 1, "token": repr(tok.decode(i)), "token_id": int(i),
             "logit": round(float(logits[i]), 2), "probability": round(float(pp), 4)} for r, (pp, i) in enumerate(zip(p, idx))]

prompts = ["The capital of India is", "Machine learning is a subset of", "I drank a cup of", "The opposite of hot is"]
table = pd.DataFrame([row for pr in prompts for row in top_k_next(pr)])
print("Probability table (top-5 next tokens):")
print(table.to_string(index=False))

# Autoregressive generation, one token at a time (greedy) - shows that generation = repeated next-token prediction
ids = tok("Generative AI is", return_tensors="pt").input_ids
for step in range(8):
    with torch.no_grad():
        nxt = model(ids).logits[0, -1].argmax()
    ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    print(f"step {step + 1}: {tok.decode(ids[0])!r}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Probability table (top-5 next tokens):
                         prompt  rank     token  token_id  logit  probability
        The capital of India is     1    ' the'       262 -97.05       0.1131
Machine learning is a subset of     1    ' the'       262 -94.54       0.1260
               I drank a cup of     1 ' coffee'      6891 -79.33       0.2740
         The opposite of hot is     1   ' cold'      4692 -69.64       0.1627
step 1: 'Generative AI is a'
step 2: 'Generative AI is a new'
step 3: 'Generative AI is a new approach'
step 4: 'Generative AI is a new approach to'
step 5: 'Generative AI is a new approach to artificial'
step 6: 'Generative AI is a new approach to artificial intelligence'
step 7: 'Generative AI is a new approach to artificial intelligence that'
step 8: 'Generative AI is a new approach to artificial intelligence that uses'
